# Data Vortex - Round 1 (Phase 1): Social Engine Data Intake Restoration
**Author:** Kislay Jha  
**Objective:** Ingest, clean, standardize corrupted social posts, and perform exploratory data analysis.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

# Yeh code automatic project root detect karega
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

raw_path = BASE_DIR / "data" / "raw"
cleaned_path = BASE_DIR / "data" / "cleaned"

print("Project Root:", BASE_DIR)
print("Looking in:", raw_path)

# Load CSV files
users = pd.read_csv(raw_path / "Social_Engine_Users.csv")
posts = pd.read_csv(raw_path / "Social_Engine_Posts_Corrupted.csv")

print(f"Success! Raw Posts: {len(posts)} | Raw Users: {len(users)}")

Project Root: c:\Users\kisla\data_vortex_round1
Looking in: c:\Users\kisla\data_vortex_round1\data\raw
Success! Raw Posts: 12360 | Raw Users: 1500


### Preprocessing & Restoration Logic
1. **Deduplication:** Dropping duplicate `post_id` entries to prevent metric inflation.
2. **Sign Inversion Correction:** Taking absolute values (`abs()`) for negative likes caused by bit-flip transmission.
3. **Median Imputation:** Filling missing likes with median likes per platform.
4. **Timestamp Parsing:** Parsing Unix epochs, ISO formats, and DD-MM-YYYY dates into standard UTC datetimes.
5. **Categorical Standardizing:** Labeling missing platforms as 'Unknown'.

In [17]:
# 1. Deduplication
posts_cleaned = posts.drop_duplicates(subset=['post_id']).copy()
users_cleaned = users.drop_duplicates(subset=['user_id']).copy()

# 2. Inverted Likes Fix (abs)
posts_cleaned['likes'] = posts_cleaned['likes'].apply(lambda x: abs(x) if pd.notnull(x) else x)

# 3. Median Imputation
platform_median_likes = posts_cleaned.groupby('platform')['likes'].transform('median')
posts_cleaned['likes'] = posts_cleaned['likes'].fillna(platform_median_likes).fillna(posts_cleaned['likes'].median()).round()

# 4. Standardize Categorical & Text Fields
posts_cleaned['platform'] = posts_cleaned['platform'].fillna('Unknown')
posts_cleaned['text_content'] = posts_cleaned['text_content'].fillna('[No Content Available]')

# 5. Timestamp Normalization
def parse_corrupted_timestamp(ts):
    if pd.isnull(ts):
        return pd.NaT
    ts_str = str(ts).strip()
    if ts_str.isdigit():
        return pd.to_datetime(int(ts_str), unit='s', errors='coerce')
    return pd.to_datetime(ts_str, dayfirst=True, errors='coerce')

posts_cleaned['cleaned_timestamp'] = posts_cleaned['timestamp'].apply(parse_corrupted_timestamp)

# 6. User Dates & Total Engagement
users_cleaned['account_created'] = pd.to_datetime(users_cleaned['account_created'], errors='coerce')
posts_cleaned['total_engagement'] = posts_cleaned['likes'] + posts_cleaned['shares'] + posts_cleaned['comments']

# 7. Cleaned folder me export
cleaned_path.mkdir(parents=True, exist_ok=True)
posts_cleaned.to_csv(cleaned_path / "Social_Engine_Posts_Cleaned.csv", index=False)
users_cleaned.to_csv(cleaned_path / "Social_Engine_Users_Cleaned.csv", index=False)

print(f"Cleaned Successfully! 12,000 posts saved to {cleaned_path}")

C:\Users\kisla\AppData\Local\Temp\ipykernel_24304\1144968317.py:23: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, dayfirst=True, errors='coerce')


Cleaned Successfully! 12,000 posts saved to c:\Users\kisla\data_vortex_round1\data\cleaned


In [18]:
merged_df = posts_cleaned.merge(users_cleaned, on='user_id', how='left')

print("1. PLATFORM SUMMARY:")
display(merged_df.groupby('platform').agg(
    post_count=('post_id', 'count'),
    mean_engagement=('total_engagement', 'mean'),
    median_engagement=('total_engagement', 'median')
).reset_index())

print("\n2. MONTHLY TIMELINE COLLAPSE:")
merged_df['year_month'] = merged_df['cleaned_timestamp'].dt.to_period('M')
display(merged_df.groupby('year_month').size().reset_index(name='post_count'))

print("\n3. TOP 5 LOCATIONS:")
display(merged_df['location'].value_counts().head(5).reset_index())

1. PLATFORM SUMMARY:


,platform,post_count,mean_engagement,median_engagement
0,Facebook,2074,4026.755545,4048.0
1,Instagram,1989,4040.833585,3993.0
2,Reddit,2031,4002.356475,4002.0
3,Twitter,2049,3949.257199,3922.0
4,Unknown,1784,3973.588565,3985.0
5,YouTube,2073,4044.551375,4084.0



2. MONTHLY TIMELINE COLLAPSE:


,year_month,post_count
0,2024-01,110
1,2024-02,110
2,2024-03,87
3,2024-04,110
4,2024-05,975
5,2024-06,923
6,2024-07,953
7,2024-08,974
8,2024-09,954
9,2024-10,965



3. TOP 5 LOCATIONS:


,location,count
0,"Los Angeles, USA",459
1,"Munich, Germany",452
2,"Shanghai, China",451
3,"Barcelona, Spain",439
4,"Houston, USA",422
